In [1]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from imblearn.under_sampling import RandomUnderSampler
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import google.generativeai as genai
from google.generativeai import types
import os
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import google.api_core.exceptions
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()



/home/shahidul/dev/project/academic/technical-debt/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
from sklearn.linear_model import LogisticRegression
class EmbeddedLogisticRegression:
    def __init__(self, model_name: str, uri:str, sentence_transformer):
        self.model_name = model_name
        self.model = LogisticRegression()
        self.uri = uri
        self.sentence_transformer = sentence_transformer

    def fit(self, x_train, y_train):
        x_train_encoded = self.sentence_transformer.encode(x_train)
        self.model.fit(x_train_encoded, y_train)

    def predict(self, x_test):
        x_test_encoded = self.sentence_transformer.encode(x_test)
        return self.model.predict(x_test_encoded)

In [4]:
# df = pd.read_csv('../data/maldonado_corrected.csv')
# df['label'] = df['satd_orig'].apply(lambda x: 'yes' if x == 1 else 'no')
# df['text'] = df['comment_text']
# df = df[["text", "label"]]
#
# under_sampler = RandomUnderSampler(sampling_strategy=1, random_state=42)
# X_resampled, y_resampled = under_sampler.fit_resample(df[['text']], df['label'])
#
# # Create balanced DataFrame
# df = pd.DataFrame({'text': X_resampled['text'], 'label': y_resampled})
# dataset = Dataset.from_pandas(df).train_test_split(test_size=0.03, seed=42)
# dataset = dataset.remove_columns(['__index_level_0__'])
# dataset

In [5]:
# df = pd.read_csv('../data/td_comment.csv')
# df['label'] = df['is_td'].apply(lambda x: 'yes' if x == 1 else 'no')
# df = df[["text", "label"]]
#
# under_sampler = RandomUnderSampler(sampling_strategy=1, random_state=42)
# X_resampled, y_resampled = under_sampler.fit_resample(df[['text']], df['label'])
#
# # Create balanced DataFrame
# df = pd.DataFrame({'text': X_resampled['text'], 'label': y_resampled})
# dataset = Dataset.from_pandas(df).train_test_split(test_size=0.2, seed=42)
# dataset = dataset.remove_columns(['__index_level_0__'])
# dataset

In [6]:
# df = pd.read_csv('../data/td_comment.csv')
# df['label'] = df['is_td'].apply(lambda x: 'yes' if x == 1 else 'no')
# df = df[["text", "label"]]
#
# under_sampler = RandomUnderSampler(sampling_strategy=1, random_state=42)
# X_resampled, y_resampled = under_sampler.fit_resample(df[['text']], df['label'])
#
# # Create balanced DataFrame
# df = pd.DataFrame({'text': X_resampled['text'], 'label': y_resampled})
# dataset = Dataset.from_pandas(df).train_test_split(test_size=0.2, seed=42)
# dataset = dataset.remove_columns(['__index_level_0__'])
# dataset

In [7]:
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')
train_dataset_all = Dataset.from_pandas(train_df)
test_dataset_all = Dataset.from_pandas(test_df)
print(train_dataset_all)
print(test_dataset_all)

Dataset({
    features: ['id', 'repository', 'comment', 'satd', 'type', 'code_before', 'code_after'],
    num_rows: 12947
})
Dataset({
    features: ['id', 'repository', 'comment', 'satd', 'type', 'code_before', 'code_after'],
    num_rows: 12947
})


In [8]:
#Manually crafted Few Shots
n_shot_df = pd.read_csv('../data/few_shot_comment.csv')
n_shot_dataset = Dataset.from_pandas(n_shot_df)
n_shot_dataset

Dataset({
    features: ['id', 'repository', 'comment', 'satd', 'type', 'code_before', 'code_after'],
    num_rows: 6
})

In [9]:
class PromptTemplate:
    def __init__(self, name, definition, instruction, n_shot_template, line_m_before, line_n_after):
        self._name = name
        self._definition = definition
        self._instruction = instruction
        self._n_shot_template = n_shot_template
        self._line_m_before = line_m_before
        self._line_n_after = line_n_after

    @property
    def name(self):
        return self._name

    @property
    def definition(self):
        return self._definition

    @property
    def instruction(self):
        return self._instruction

    @property
    def line_m_before(self):
        return self._line_m_before

    @property
    def line_n_after(self):
        return self._line_n_after

    @property
    def shot_template(self):
        return self._n_shot_template

    def __repr__(self):
        return f"PromptTemplate(name={self.name}, description='{self.definition}', example='{self.shot_template}')"

In [10]:
class ModelConfig:
    def __init__(self, name: str, architecture: str, uri: str):
        self.name = name
        self.architecture = architecture
        self.uri = uri

    def __repr__(self):
        return f"ModelConfig(name='{self.name}', uri='{self.uri}')"

In [11]:
class ChatGpt4Model:
    def __init__(self, model_name: str, client: OpenAI):
        self.model_name = model_name
        self.client = client

    def generate(self, prompt):
        completion = self.client.chat.completions.create(
            model=self.model_name,
            store=True,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return completion.choices[0].message.content.strip().split()[-1].lower()


In [12]:
import jpype
import jpype.imports
from jpype.types import *
from dotenv import load_dotenv
import os
load_dotenv()


class TextMiningBasedSatdDetector:
    def __init__(self, model_name: str):
        self.model_name = model_name
    def fit(self, x_train, y_train):
        pass
    def predict(self, x_test):
        if not jpype.isJVMStarted():
            jar_path = os.getenv('SATD_DETECTOR_JAR')
            dependency_path= os.getenv('SATD_DETECTOR_DEPENDENCY')
            jvm_args = ["-Xss512m"]
            jpype.startJVM(jpype.getDefaultJVMPath(),  classpath=[jar_path, dependency_path], )
        from satd_detector.core.utils import SATDDetector
        detector1 = SATDDetector()
        y_pred = []
        for comment in x_test:
            if detector1.isSATD(comment):
                y_pred.append('yes')
            else:
                y_pred.append('no')
        return y_pred



In [53]:
class GeminiSentenceTransformer:
    def __init__(self, uri: str, use_cache=False):
        self.uri = uri
        self.use_cache = use_cache
        file = f'./cache/{uri.split("/")[-1]}.csv'
        if not os.path.exists(file):
              with open(file, "w") as file:
                file.write_text("text,embedding")
        self.cache_df = pd.read_csv(file)
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
    def encode(self, features):
        encoded_features = []
        for text in features:
            if self.use_cache:
                embedding_df = self.cache_df[self.cache_df[self.cache_df['text'] == text]]['embedding']
                if embedding_df.empty:
                    response = genai.embed_content(
                    model=self.uri,
                    content=text,
                    task_type="classification")
                    embedding_value = np.array(response["embedding"])
                    self.cache_df.loc[len(self.cache_df)] = [text, embedding_value]
                    self.cache_df.to_csv(f'./cache/{self.uri.split("/")[-1]}.csv', index=False)
                    encoded_features.append(embedding_value)
                else:
                    encoded_features.append(embedding_df.iloc[0])
            else:
                raise Exception('Not Implemented Yet')
        return encoded_features

In [14]:
from util import get_first_n_line, get_last_n_line


def create_prompt(prompt_template, n_shot_x, n_shot_y, n_shot_code_before, n_shot_code_after, question, code_before, code_after,
                  tokenizer):
    instances = []
    for index, [x, y, cb, ca] in enumerate(zip(n_shot_x, n_shot_y, n_shot_code_before, n_shot_code_after)):
        example_text = prompt_template.shot_template.format(**{'comment': x,
                                                             'label': y,
                                                             'code_before': get_last_n_line(cb, prompt_template.line_m_before), 'code_after': get_first_n_line(ca, prompt_template.line_n_after)})
        instances.append(example_text)

    instances.append(prompt_template.shot_template.format(**{'comment': question,
                                                           'label': '',
                                                           'code_before': get_last_n_line(code_before, prompt_template.line_m_before), 'code_after': get_first_n_line(code_after, prompt_template.line_n_after)}))
    prompt_text = prompt_template.definition + "\n" + prompt_template.instruction + "\n" + "\n" + "\n\n".join(instances)
    return prompt_text


In [15]:
# from util import get_first_n_line, get_last_n_line
# def get_token_length(tokenizer, text):
#     if tokenizer:
#         tokens = tokenizer(text, return_tensors='pt')
#         return tokens['input_ids'].shape[1]
#     else:
#         return len(text)
#
#
# def create_prompt(prompt_template, x, y, n_shot_code_before, n_shot_code_after, question, code_before, code_after,
#                   tokenizer):
#     model_max_length = tokenizer.model_max_length if tokenizer else float('inf')
#     initial_text = prompt_template.definition + "\n" + prompt_template.instruction
#     allocated_tokens = len(x) + 5  # Formatting
#     allocated_tokens += get_token_length(tokenizer, initial_text)
#     question_prefix = question
#     while len(question_prefix) > 0:
#         target_sample_token_length = get_token_length(tokenizer,
#                                                       prompt_template.shot_template.format(question_prefix, '', get_last_n_line(code_before, prompt_template.line_m_before), get_first_n_line(code_after, prompt_template.line_n_after)))
#         if allocated_tokens + target_sample_token_length <= model_max_length:
#             allocated_tokens += target_sample_token_length
#             break
#         else:
#             question_prefix = question_prefix[: len(question_prefix) // 2]
#
#     instances = []
#     skipped = 0
#     for index, [x, y, cb, ca] in enumerate(zip(x, y, n_shot_code_before, n_shot_code_after)):
#         example_text = prompt_template.shot_template.format(x, y, get_last_n_line(cb, prompt_template.line_m_before), get_first_n_line(ca, prompt_template.line_n_after))
#         example_token_length = get_token_length(tokenizer, example_text)
#         if allocated_tokens + example_token_length < model_max_length:
#             instances.append(prompt_template.shot_template.format(x, y, get_last_n_line(cb, prompt_template.line_m_before), get_first_n_line(ca, prompt_template.line_n_after)))
#             allocated_tokens += example_token_length
#         else:
#             skipped += 1
#     if skipped > 0:
#         print(f'Skipping {skipped} shots due to token limits')
#
#     instances.append(prompt_template.shot_template.format(question_prefix, '', get_last_n_line(code_before, prompt_template.line_m_before), get_first_n_line(code_after, prompt_template.line_n_after)))
#     prompt_text = initial_text + "\n" + "\n" + "\n\n".join(instances)
#     prompt_token_length = get_token_length(tokenizer, prompt_text)
#     if prompt_token_length > model_max_length:
#         print(f'prompt length {prompt_token_length}')
#     return prompt_text

In [38]:
def create_model_and_tokenizer(model_config: ModelConfig):
    is_hf_model = True
    uri = model_config.uri
    if 'sentence-embedded-regression' in model_config.architecture.lower():
        return EmbeddedLogisticRegression(model_config.name, model_config.uri, SentenceTransformer(uri)), None
    elif 'gemini-embedded-regression' in model_config.architecture.lower():
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
        return EmbeddedLogisticRegression(model_config.name, model_config.uri, GeminiSentenceTransformer(uri, True)), None
    elif 'text-mining' in model_config.architecture.lower():
        return TextMiningBasedSatdDetector(model_config.uri), None
    elif 'gpt-4' in model_config.architecture:
        return ChatGpt4Model(model_config.uri, OpenAI(api_key=os.getenv("OPEN_AI_API_KEY"))), None
    if 'gpt' in model_config.architecture.lower():
        model = AutoModelForCausalLM.from_pretrained(uri)
    elif "/bert" in model_config.architecture.lower() or "/codebert" in model_config.architecture.lower():
        model = AutoModelForSequenceClassification.from_pretrained(uri, num_labels=2)
    elif 'gemini' in model_config.architecture.lower():
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
        model = genai.GenerativeModel(uri)
        is_hf_model = False
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(uri)
    if is_hf_model:
        model.to(device)
        tokenizer = AutoTokenizer.from_pretrained(model_config.uri, use_fast=True)
    else:
        tokenizer = None
    return model, tokenizer


In [17]:
from enum import Enum


class FewShotSelectionStrategy(Enum):
    RANDOM = 'random'
    SIMILAR = 'similar'
    MANUAL_CRAFTED = 'manual_crafted'


In [18]:
import random


def pick_n_shot(train_dataset, x, y, index, st_similarity, n=0, strategy=None):
    if len(x) < n:
        raise Exception(f'only {len(x)} examples available for {n} shots')
    indexes = []
    if strategy == FewShotSelectionStrategy.RANDOM:
        indexes = random.sample(range(len(x)), n)
    elif strategy == FewShotSelectionStrategy.SIMILAR:
        _, top_n_indices = st_similarity[index].topk(n)
        indexes.extend(top_n_indices.tolist())
    elif strategy == FewShotSelectionStrategy.MANUAL_CRAFTED:
        indexes = [i for i in range(n)]
    shot_x = []
    shot_y = []
    code_before = []
    code_after = []
    for index in indexes:
        shot_x.append(x[index])
        shot_y.append(y[index])
        code_before.append(train_dataset['code_before'][index])
        code_after.append(train_dataset['code_after'][index])
    return shot_x, shot_y, code_before, code_after


In [19]:
sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')


In [20]:
def print_classification(test_x, test_y, y_pred):
    print("Incorrect Predictions:")
    for text, true, pred in zip(test_x, test_y, y_pred):
        if true != pred:
            print(f"✖ {text} (Label: {true}, Predicted: {pred})")


In [21]:
# PROMPT_TEMPLATES = [
#     PromptTemplate(
#         name="With Indicators",
#         definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that solely describe expected behavior, general actions, testing actions or issue references unless they explicitly acknowledge it requires future work.",
#         instruction="Classify whether the comment contains SATD (yes/no)",
#         n_shot_template="<EXAMPLE>\nComment: {}\nLabel: {}\n</EXAMPLE>"
#     ),
#     PromptTemplate(
#         name="No Keywords",
#         definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments.",
#         instruction="Assign the label of yes or no for each given source code comment",
#         n_shot_template="Comment: {}\nLabel: {}"
#     ),
#     PromptTemplate(
#         name="MAT Keywords",
#         definition="Self-admitted technical debt (SATD) are technical debt admitted by the developer through source code comments. SATD comments usually  contain specific keywords: TODO, FIXME, HACK, and XXX.",
#         instruction="Assign the label of yes or no for each given source code comment.",
#         n_shot_template="Comment: {}\nLabel: {}"
#     ), PromptTemplate(
#         name="Jitterbug Keywords",
#         definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. SATD comments usually contain specific keywords: TODO, FIXME, HACK, and Workaround.",
#         instruction="Assign the label of yes or no for each given source code comment.",
#         n_shot_template="Comment: {}\nLabel: {}"
#     ), PromptTemplate(
#         name="GPT4 Keywords",
#         definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. SATD comments usually contain specific keywords: TODO, FIXME, HACK, XXX, NOTE, DEBT, REFACTOR, OPTIMIZE, TEMP, WORKAROUND, KLUDGE, REVIEW, NOFIX, PENDING, and BUG.",
#         instruction="Assign the label of yes or no for each given source code comment.",
#         n_shot_template="Comment: {}\nLabel: {}"
#     ), PromptTemplate(
#         name="TD",
#         definition="Technical debt (TD) in code comment are comments that that indicates weak code or something need to be done. TD comments usually  contain specific keywords: TODO, FIXME, HACK, and XXX.",
#         instruction="Assign the label of yes or no for each given source code comment.",
#         n_shot_template="Comment: {}\nLabel: {}"
#     )]

In [22]:
FEW_SHOT_SIZES = [0, 1, 2, 3, 5, 10, 15, 20]
MODEL_CONFIG_GEMINI_2_FLASH = ModelConfig(name="Gemini 2 Flash", architecture="gemini", uri="models/gemini-2.0-flash")
MODEL_CONFIG_SENTENCE_EMBEDDED_LR = ModelConfig(name="Sentence Embedded Logistic Regression", architecture="sentence-embedded-regression", uri="all-MiniLM-L6-v2")
MODEL_CONFIG_GEMINI_EMBEDDED_LR = ModelConfig(name="Gemini Embedded Logistic Regression", architecture="gemini-embedded-regression", uri='models/gemini-embedding-exp-03-07')
MODEL_CONFIG_TEXT_MINING_SATD_DETECTOR = ModelConfig(name="Text Mining SATD Detector", architecture="text-mining", uri="text-mining/satd-detector")
MODEL_CONFIGS = [
    MODEL_CONFIG_GEMINI_2_FLASH,
    ModelConfig(name="Chat GPT 4o mini", architecture="gpt-4", uri="gpt-4o-mini"),
    ModelConfig(name="Chat GPT 4o", architecture="gpt-4", uri="gpt-4o"),
    ModelConfig(name="Flan T5 Small", architecture='flan-t5', uri="google/flan-t5-small"),
    ModelConfig(name="Flan T5 Base", architecture='flan-t5', uri="google/flan-t5-base"),
    ModelConfig(name="Flan T5 Large", architecture='flan-t5', uri="google/flan-t5-large"),
    ModelConfig(name="Flan T5 XL", architecture='flan-t5', uri="google/flan-t5-xl"),
    ModelConfig(name="BERT Base", architecture="bert", uri="google-bert/bert-base-uncased"),
    ModelConfig(name="CodeBERT Base", architecture="codebert", uri="microsoft/codebert-base"),
    ModelConfig(name="Facebook BART Base", architecture="bart", uri="facebook/bart-base")

]
FEW_SHOT_STRATEGIES = [FewShotSelectionStrategy.MANUAL_CRAFTED, FewShotSelectionStrategy.RANDOM,
                       FewShotSelectionStrategy.SIMILAR]

In [23]:
def predict_with_prompt(model, tokenizer, prompt):
    if isinstance(model, ChatGpt4Model):
        return model.generate(prompt)
    elif tokenizer:
        inputs = tokenizer(prompt, return_tensors='pt')
        inputs = {key: value.to(device) for key, value in inputs.items()}
        output = tokenizer.decode(
            model.generate(
                inputs["input_ids"],
                max_new_tokens=50
                # generation_config=GenerationConfig(max_new_tokens=5, do_sample=True, temperature=0.01)
            )[0],
            skip_special_tokens=True
        )
        return output.strip()
    else:
        return predict_with_gemini(model, prompt)


@retry(
    stop=stop_after_attempt(10),  # Stop after 5 retries
    wait=wait_exponential(multiplier=2, min=60, max=2 * 60),
    retry=retry_if_exception_type(google.api_core.exceptions.ResourceExhausted),  # Retry on rate limit errors
)
def predict_with_gemini(model, prompt):
    generation_config = types.GenerationConfig(
        temperature=0.0

    )
    return model.generate_content(contents=prompt, generation_config=generation_config).text.split()[-1].lower()


In [24]:
from datetime import datetime
LAST_RUNNING_FILE = None


def detect_satd(model_config, few_shot_size, prompt_template, few_shot_strategy, dataset, text_column='text', label_column='label', verbose = False):
    train_dataset = dataset["train"]
    train_x = train_dataset[text_column]
    train_y = train_dataset[label_column]
    test_dataset = dataset["test"]
    test_x = test_dataset[text_column]
    test_y = test_dataset[label_column]
    test_x_code_before = test_dataset['code_before']
    test_x_code_after = test_dataset['code_after']
    st_similarities = cos_sim(sentence_transformer.encode(test_x),
                                                  sentence_transformer.encode(train_x)) if FewShotSelectionStrategy.SIMILAR == few_shot_strategy else None
    model, tokenizer = create_model_and_tokenizer(model_config)
    y_pred = []
    unseen_labels = []
    running_config = f'{model_config.name} - {prompt_template.name} {few_shot_size} -  {few_shot_strategy}'
    print(f'Running {running_config}')
    if isinstance(model, EmbeddedLogisticRegression) or isinstance(model, TextMiningBasedSatdDetector):
        model.fit(train_x, train_y)
        y_pred = model.predict( test_x)
    else:
        for i, [text, label, code_before, code_after] in enumerate(zip(test_x, test_y, test_x_code_before, test_x_code_after)):
            shot_x, shot_y, n_shot_code_before, n_shot_code_after = pick_n_shot(train_dataset, train_x, train_y, i, st_similarities, few_shot_size,
                                         few_shot_strategy)
            prompt = create_prompt(prompt_template, shot_x, shot_y, n_shot_code_before, n_shot_code_after, text,
                                   code_before, code_after, tokenizer)
            pred = predict_with_prompt(model, tokenizer, prompt)
            if verbose:
                print(f'Prompt\n {prompt}')
                if pred != label:
                    print(f'{pred} {label} Failing for {text}')
            if pred not in ['yes', 'no']:
                unseen_labels.append(pred)
                pred = 'no'
            y_pred.append(pred)
    test_output = dataset['test'].to_dict()
    test_output[label_column + '_pred'] = y_pred
    print(classification_report(test_y, y_pred, zero_division=0, digits=3))
    if verbose:
        print(f'Unknown classification count {len(unseen_labels)}')
        print(f'Unknown classification  {unseen_labels}')
    timestamp = datetime.now().strftime("%B %d, %Y, %H:%M:%S")
    global LAST_RUNNING_FILE
    LAST_RUNNING_FILE = f'./cache/{timestamp}_{running_config}.csv'

    Dataset.from_dict(test_output).to_pandas().to_csv(LAST_RUNNING_FILE, index = False)
    # return test_y, y_pred

In [25]:
template = PromptTemplate(
        name="Manually Crafted",
        definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
        instruction="Assign the label yes if the comment contains a strong indication of Self-Admitted Technical Debt (SATD); otherwise, assign the label no.",
        n_shot_template="<EXAMPLE>\nComment: {comment}\nLabel: {label}\nCode Before Comment: {code_before}\nCode After Comment: {code_after}\n</EXAMPLE>",
        line_m_before=3,
        line_n_after=10
    )

# template = PromptTemplate(
#         name="Manually Crafted",
#         definition="You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
#         instruction="Classify whether the comment contains SATD if the confidence level is high(yes/no)",
#         n_shot_template="<EXAMPLE>\nComment: {comment}\nLabel: {label}\nCode Before Comment: {code_before}\nCode After Comment: {code_after}\n</EXAMPLE>",
#         line_m_before=3,
#         line_n_after=3
#     )


In [30]:
# data_frame = pd.read_csv('../data/test.csv')
data_frame = pd.read_csv('./cache/comment_mismatch.csv')
from_row_id, to_row_id = [int(index) if index else None for index in input('Row ID').strip().split(':')]
data_frame = data_frame[from_row_id:to_row_id]

input_test_dataset = Dataset.from_pandas(data_frame)

In [28]:
#LLM
detect_satd(MODEL_CONFIG_GEMINI_2_FLASH,4, template, FewShotSelectionStrategy.MANUAL_CRAFTED,  DatasetDict({'train': n_shot_dataset, 'test': input_test_dataset}), text_column = 'comment', label_column = 'satd', verbose = True)


Running Gemini 2 Flash - Manually Crafted 4 -  FewShotSelectionStrategy.MANUAL_CRAFTED
Prompt
 You are an AI model trained to detect Self-Admitted Technical Debt (SATD) in software development test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional context indicating suboptimal code that require future work. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).
Assign the l

In [113]:
#Logistic Regression
# under_sampler = RandomUnderSampler(sampling_strategy='auto', random_state=42)
# lr_df_all = train_dataset_all.to_pandas()
# indices, _ = under_sampler.fit_resample(lr_df_all.index.values.reshape(-1, 1), lr_df_all['satd'])
# lr_df_resampled = lr_df_all.loc[indices.flatten()]
# lr_df_resampled.reset_index(drop=True)
# lr_dataset = Dataset.from_pandas(lr_df_resampled)

sentence_embedded_lg_train_df = pd.read_csv('../data/train_balanced.csv')
lr_dataset = Dataset.from_pandas(sentence_embedded_lg_train_df)
detect_satd(MODEL_CONFIG_SENTENCE_EMBEDDED_LR, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED, DatasetDict({'train': lr_dataset, 'test': input_test_dataset}), text_column ='comment', label_column ='satd', verbose = True)

Running Sentence Embedded Logistic Regression - Manually Crafted 0 -  FewShotSelectionStrategy.MANUAL_CRAFTED
              precision    recall  f1-score   support

          no      1.000     1.000     1.000         1

    accuracy                          1.000         1
   macro avg      1.000     1.000     1.000         1
weighted avg      1.000     1.000     1.000         1

Unknown classification count 0
Unknown classification  []


In [43]:
#Logistic Regression with Gemini
# genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
gemini_lg_train_df = pd.read_csv('../data/train_balanced.csv')
gemini_lg_train_dataset = Dataset.from_pandas(gemini_lg_train_df)
detect_satd(MODEL_CONFIG_GEMINI_EMBEDDED_LR, 0, template, FewShotSelectionStrategy.MANUAL_CRAFTED, DatasetDict({'train': gemini_lg_train_dataset, 'test': input_test_dataset}), text_column ='comment', label_column ='satd', verbose = True)

Running Gemini Embedded Logistic Regression - Manually Crafted 0 -  FewShotSelectionStrategy.MANUAL_CRAFTED


ValueError: ('Lengths must match to compare', (0,), (584,))

In [36]:
LAST_RUNNING_FILE ='./cache/March 18, 2025, 23:01:49_Gemini 2 Flash - Manually Crafted 4 -  FewShotSelectionStrategy.MANUAL_CRAFTED.csv'
if LAST_RUNNING_FILE:
    print('Merging Mismatch')
    ldf = pd.read_csv(LAST_RUNNING_FILE)
    mdf = pd.read_csv('./cache/comment_output.csv')
    ids = mdf['id'].values
    for index, row in ldf.iterrows():
        if row['id'] in ids:
            mdf.loc[mdf['id'] == row['id'], 'satd_pred'] = row['satd_pred']
        else:
            mdf.loc[len(mdf)] = row
    mdf.sort_values(by=['id'], ascending=True, inplace=True)
    mdf.to_csv('./cache/comment_output.csv', index = False)
    mdf[mdf['satd'] != mdf['satd_pred']].to_csv('./cache/comment_mismatch.csv', index = False)

Merging Mismatch


In [28]:
# from comment import CommentRepository
# from db_config import SessionLocal
# from dotenv import load_dotenv
# import os
#
# load_dotenv()
# session = SessionLocal()
# repo = CommentRepository()
# comments = repo.get_comments_with_no_prediction(limit=500)
# ids = []
# texts = []
# labels = []
# for comment in comments:
#     ids.append(comment.id)
#     texts.append(comment.text)
#     labels.append(comment.is_td)
#
# test_dataset = Dataset.from_dict({'text': texts, 'label': labels})
# detection_dataset = DatasetDict({
#     'train': dataset['train'],
#     'test': test_dataset
# })
# # print(detection_dataset)
#
# output = detect_satd(MODEL_CONFIGS[0:1], [3], PROMPT_TEMPLATES[0:1], [FewShotSelectionStrategy.MANUAL_CRAFTED], detection_dataset)
# rc, test_x, test_y, pred_y, unknown_labels = output[0]
# for _,[id, pred] in enumerate(zip(ids,  pred_y)):
#     target_comment = repo.get_comment(id)
#     target_comment.pred_td = True if pred.lower() == 'yes' else False
#     session.merge(target_comment)
#     session.commit()


In [29]:

# target_model = 'gemini'
# comment_df = pd.read_csv('../data/comments.csv')
# test_df = comment_df[comment_df[target_model] is None].sample(frac=1, random_state=42).head(10)
#
# ids = []
# texts = []
# labels = []
# for index, row in df.iterrows():
#     ids.append(row['id'])
#     texts.append(row['text'])
#     labels.append(row[target_model])
#
# test_dataset = Dataset.from_dict({'text': texts, 'label': labels})
# detection_dataset = DatasetDict({
#     'train': dataset['train'],
#     'test': test_dataset
# })
#
# output = detect_satd(MODEL_CONFIGS[0:1], [20], PROMPT_TEMPLATES[0:1], [FewShotSelectionStrategy.MANUAL_CRAFTED],
#                      detection_dataset)
# rc, test_x, test_y, pred_y, unknown_labels = output[0]
# for _, [id, pred] in enumerate(zip(ids, pred_y)):
#     comment_df.loc[df['id'] == id, target_model] = pred
# comment_df.to_csv('../data/comments.csv')
#
